# Bio_ClinicalBERT — Extractor de características congelado sobre Notas de Radiología

Uso de **Bio_ClinicalBERT** (`emilyalsentzer/Bio_ClinicalBERT`) como **extractor de características congelado** para predicción binaria de sepsis a partir de informes de radiología.

**Estrategia**: para cada stay, se usa el informe de radiología más reciente **anterior** al tiempo de referencia (`ref_hour`). Esto evita data leakage: el modelo solo ve información disponible antes del horizonte de predicción.

**Enfoque**: todos los parámetros de Bio_ClinicalBERT se mantienen **congelados** (sin fine-tuning). Se extrae el embedding `[CLS]` (768-dim) y se entrena únicamente un MLP ligero (768→256→64→1) encima. Se descartó el fine-tuning de los pesos de BERT por su coste computacional (entrenamiento en CPU).

**Salida**: MLP entrenado para 6h y 12h + embeddings `[CLS]` del train/val/test para la red de fusión.

In [1]:
import os
# GPU habilitada (CUDA 12.8): ya no se oculta el dispositivo

import polars as pl
import numpy as np
import json
import pickle
from pathlib import Path
import time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModel,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import matplotlib.pyplot as plt

NOTES_PATH = Path.home() / "mimic-iv-note" / "note" / "radiology.csv.gz"
MIMIC      = Path.home() / "mimic-iv-3.0"
PROC       = Path("../data/processed")
OUT_DIR    = Path("../experiments/clinicalbert")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME        = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LEN           = 128    # 512→128: 16x más rápido en atención; info clave está al inicio
MAX_TRAIN_SAMPLES = 8000   # subsample estratificado del train set
BATCH_SIZE        = 32     # más grande porque MAX_LEN es menor
GRAD_ACCUM        = 2      # batch efectivo = 64
LR                = 2e-5
N_EPOCHS          = 5
PATIENCE          = 2
SEED              = 42
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

print(f"Device: {DEVICE}  |  CUDA visible: {torch.cuda.is_available()}")
print(f"Modelo: {MODEL_NAME}  |  MAX_LEN={MAX_LEN}  BS={BATCH_SIZE}×{GRAD_ACCUM}  MAX_TRAIN={MAX_TRAIN_SAMPLES}")

Device: cuda  |  CUDA visible: True
Modelo: emilyalsentzer/Bio_ClinicalBERT  |  MAX_LEN=128  BS=32×2  MAX_TRAIN=8000


## PASO 1 — Cargar cohorte y asignar ref_hour

In [2]:
log("Cargando cohorte ...")

cohort = pl.read_parquet(PROC / "cohort.parquet").select([
    "subject_id", "hadm_id", "stay_id", "los_hours", "sepsis", "intime",
])
splits = pl.read_parquet(PROC / "splits.parquet").select(["stay_id", "split"])
cohort = cohort.join(splits, on="stay_id", how="left")

vitals_meta = (
    pl.read_parquet(PROC / "vitals/vitals_hourly.parquet",
                    columns=["stay_id", "onset_hour"])
    .unique(subset=["stay_id"])
)
cohort = cohort.join(vitals_meta, on="stay_id", how="left")

# intime como datetime
cohort = cohort.with_columns(
    pl.col("intime").cast(pl.Datetime("us"))
)

print(f"Cohorte: {len(cohort):,} stays")

def assign_ref_hour(cohort, horizon, seed):
    rng = np.random.default_rng(seed)
    out = []
    for r in cohort.to_dicts():
        if r["sepsis"] == 1:
            ref = (int(r["onset_hour"]) - horizon) if r["onset_hour"] is not None else None
        else:
            lo, hi = 24, int(r["los_hours"]) - horizon
            ref = int(rng.integers(lo, hi + 1)) if hi >= lo else None
        if ref is not None and ref >= 12:
            r["ref_hour"] = ref
            out.append(r)
    return pl.DataFrame(out)

cohort_6h  = assign_ref_hour(cohort, 6,  SEED)
cohort_12h = assign_ref_hour(cohort, 12, SEED + 1)

print(f"Stays válidos 6h:  {len(cohort_6h):,}")
print(f"Stays válidos 12h: {len(cohort_12h):,}")

[10:57:55] Cargando cohorte ...
Cohorte: 74,829 stays


Stays válidos 6h:  53,453
Stays válidos 12h: 47,219


## PASO 2 — Cargar notas de radiología y enlazar con stays

Las notas tienen `charttime` (timestamp durante la estancia) y `hadm_id`.  
Para cada stay seleccionamos el informe más reciente **antes** del `ref_time = intime + ref_hour`.  
Stays sin nota quedan excluidos del entrenamiento BERT (se manejan en la fusión con embedding nulo).

In [3]:
log("Cargando notas de radiología (puede tardar 1-2 min) ...")

notes = pl.read_csv(
    NOTES_PATH,
    columns=["subject_id", "hadm_id", "charttime", "text"],
    infer_schema_length=10000,
).with_columns([
    pl.col("subject_id").cast(pl.Int64),
    pl.col("hadm_id").cast(pl.Int64),
    pl.col("charttime").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False),
])

# Filtrar solo hadm_ids de la cohorte
cohort_hadm = cohort["hadm_id"].unique().to_list()
notes = notes.filter(pl.col("hadm_id").is_in(cohort_hadm))

log(f"Notas tras filtro de cohorte: {len(notes):,}")


def assign_notes(cohort_h, notes):
    """
    Para cada stay, devuelve el texto de la nota más reciente antes de ref_time.
    ref_time = intime + ref_hour horas.
    """
    cohort_with_ref = cohort_h.with_columns(
        (pl.col("intime") + pl.duration(hours=pl.col("ref_hour"))).alias("ref_time")
    ).select(["stay_id", "hadm_id", "ref_time", "sepsis", "split"])

    joined = (
        cohort_with_ref
        .join(notes.select(["hadm_id", "charttime", "text"]), on="hadm_id", how="left")
        .filter(pl.col("charttime") <= pl.col("ref_time"))
        .sort(["stay_id", "charttime"])
        .group_by("stay_id")
        .agg([
            pl.col("text").last().alias("note_text"),
            pl.col("charttime").last().alias("note_charttime"),
            pl.col("sepsis").first(),
            pl.col("split").first(),
        ])
        .filter(pl.col("note_text").is_not_null())
    )
    return joined

log("Asignando notas a stays 6h ...")
notes_6h  = assign_notes(cohort_6h,  notes)
log("Asignando notas a stays 12h ...")
notes_12h = assign_notes(cohort_12h, notes)

for h, df in [("6h", notes_6h), ("12h", notes_12h)]:
    n_total = len(df)
    n_sep   = df["sepsis"].sum()
    print(f"  {h}: {n_total:,} stays con nota  |  sepsis: {n_sep:,} ({100*n_sep/n_total:.1f}%)")

[10:57:56] Cargando notas de radiología (puede tardar 1-2 min) ...


[10:58:00] Notas tras filtro de cohorte: 461,915
[10:58:00] Asignando notas a stays 6h ...
[10:58:00] Asignando notas a stays 12h ...
  6h: 40,588 stays con nota  |  sepsis: 2,820 (6.9%)
  12h: 35,971 stays con nota  |  sepsis: 2,554 (7.1%)


## PASO 3 — Tokenización y Dataset

In [4]:
log(f"Cargando tokenizer {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
log("Tokenizer cargado.")


def stratified_subsample(df, label_col, n, seed):
    """Subsample estratificado manteniendo la proporción de clases."""
    rng = np.random.default_rng(seed)
    pos = df.filter(pl.col(label_col) == 1)
    neg = df.filter(pl.col(label_col) == 0)
    prev = len(pos) / len(df)
    n_pos = min(int(n * prev), len(pos))
    n_neg = min(n - n_pos, len(neg))
    idx_pos = rng.choice(len(pos), n_pos, replace=False)
    idx_neg = rng.choice(len(neg), n_neg, replace=False)
    return pl.concat([pos[idx_pos], neg[idx_neg]])


class RadiologyDataset(Dataset):
    def __init__(self, notes_df, split_name, tokenizer, max_len,
                 max_train_samples=None):
        subset = notes_df.filter(pl.col("split") == split_name)

        if split_name == "train" and max_train_samples and len(subset) > max_train_samples:
            subset = stratified_subsample(subset, "sepsis", max_train_samples, SEED)
            log(f"    Train subsampled: {len(subset):,} muestras (estratificado)")

        texts  = subset["note_text"].to_list()
        labels = subset["sepsis"].to_list()

        log(f"    Tokenizando {len(texts):,} notas ({split_name}) ...")
        enc = tokenizer(
            texts,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        self.input_ids      = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels         = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_mask[idx], self.labels[idx]


def make_loaders(notes_df, tokenizer, max_len, batch_size, max_train_samples=None):
    loaders = {}
    for split in ["train", "val", "test"]:
        ds = RadiologyDataset(notes_df, split, tokenizer, max_len,
                              max_train_samples if split == "train" else None)
        loaders[split] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=0,
        )
        print(f"      {split}: {len(ds):,} muestras")
    return loaders

log("Construyendo dataloaders 6h ...")
loaders_6h  = make_loaders(notes_6h,  tokenizer, MAX_LEN, BATCH_SIZE, MAX_TRAIN_SAMPLES)
log("Construyendo dataloaders 12h ...")
loaders_12h = make_loaders(notes_12h, tokenizer, MAX_LEN, BATCH_SIZE, MAX_TRAIN_SAMPLES)

[10:58:00] Cargando tokenizer emilyalsentzer/Bio_ClinicalBERT ...


/home/fer/miniconda3/envs/sepsis-tfm/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[10:58:01] Tokenizer cargado.
[10:58:01] Construyendo dataloaders 6h ...
[10:58:01]     Train subsampled: 8,000 muestras (estratificado)
[10:58:01]     Tokenizando 8,000 notas (train) ...


      train: 8,000 muestras
[10:58:01]     Tokenizando 6,132 notas (val) ...


      val: 6,132 muestras
[10:58:02]     Tokenizando 6,050 notas (test) ...


      test: 6,050 muestras
[10:58:03] Construyendo dataloaders 12h ...
[10:58:03]     Train subsampled: 8,000 muestras (estratificado)
[10:58:03]     Tokenizando 8,000 notas (train) ...


      train: 8,000 muestras
[10:58:03]     Tokenizando 5,420 notas (val) ...


      val: 5,420 muestras
[10:58:04]     Tokenizando 5,376 notas (test) ...


      test: 5,376 muestras


## PASO 4 — Modelo: Bio_ClinicalBERT congelado + MLP

**Todos** los parámetros del Transformer (12 capas) se mantienen congelados (`requires_grad = False`); Bio_ClinicalBERT actúa solo como extractor del embedding `[CLS]`.

Se entrena únicamente un clasificador MLP (768 → 256 → 64 → 1) sobre los embeddings `[CLS]` extraídos. Este enfoque de extractor congelado evita el coste de retropropagar a través de los 110M parámetros de BERT y hace viable el entrenamiento en CPU.

In [5]:
class ClinicalBertExtractor(nn.Module):
    """ClinicalBERT completamente congelado — solo extrae embeddings [CLS]."""
    def __init__(self, model_name):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]   # [B, 768]


class SepsisMLPClassifier(nn.Module):
    """MLP entrenado sobre embeddings [CLS] de ClinicalBERT."""
    def __init__(self, input_dim=768, hidden_dims=(256, 64), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


log(f"Cargando {MODEL_NAME} (frozen) ...")
extractor = ClinicalBertExtractor(MODEL_NAME).to(DEVICE)
n_params  = sum(p.numel() for p in extractor.parameters())
print(f"BERT params (todos congelados): {n_params:,}")
print(f"MLP params (entrenables):       {sum(p.numel() for p in SepsisMLPClassifier().parameters()):,}")

[10:58:04] Cargando emilyalsentzer/Bio_ClinicalBERT (frozen) ...


BERT params (todos congelados): 108,310,272
MLP params (entrenables):       214,017


## PASO 5 — Entrenamiento

In [6]:
from torch.utils.data import TensorDataset

def extract_embeddings(extractor, loader, desc=""):
    """Forward pass sin gradientes — devuelve (embeddings, labels) como tensors."""
    extractor.eval()
    all_emb, all_lbl = [], []
    n_batches = len(loader)
    for i, (input_ids, attn_mask, labels) in enumerate(loader):
        if i % 20 == 0:
            print(f"    {desc} {i}/{n_batches} batches ...", end="\r")
        emb = extractor(input_ids.to(DEVICE), attn_mask.to(DEVICE))
        all_emb.append(emb.cpu())
        all_lbl.append(labels)
    print()
    return torch.cat(all_emb), torch.cat(all_lbl)


def train_mlp(emb_tr, lbl_tr, emb_va, lbl_va, pos_weight_val,
              n_epochs=100, patience=10, lr=1e-3, batch_size=256):
    ds_tr = TensorDataset(emb_tr, lbl_tr)
    ds_va = TensorDataset(emb_va, lbl_va)
    ld_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True)
    ld_va = DataLoader(ds_va, batch_size=batch_size)

    mlp = SepsisMLPClassifier().to(DEVICE)
    pos_w = torch.tensor([pos_weight_val], dtype=torch.float32).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=4, factor=0.5
    )

    best_auroc, best_state, patience_count = 0.0, None, 0

    for epoch in range(1, n_epochs + 1):
        mlp.train()
        for emb_b, lbl_b in ld_tr:
            emb_b, lbl_b = emb_b.to(DEVICE), lbl_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(mlp(emb_b), lbl_b.float())
            loss.backward()
            optimizer.step()

        mlp.eval()
        with torch.no_grad():
            val_logits = mlp(emb_va.to(DEVICE)).cpu().numpy()
        val_probs  = torch.sigmoid(torch.tensor(val_logits)).numpy()
        val_auroc  = roc_auc_score(lbl_va.numpy(), val_probs)
        scheduler.step(val_auroc)

        if epoch % 10 == 0:
            print(f"    Epoch {epoch:3d} | val AUROC={val_auroc:.4f}")

        if val_auroc > best_auroc:
            best_auroc = val_auroc
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"    Early stopping en epoch {epoch}")
                break

    mlp.load_state_dict(best_state)
    print(f"    Mejor val AUROC: {best_auroc:.4f}")
    return mlp


# --- Paso 1: extraer embeddings (forward pass único, sin gradientes) ---
log("Extrayendo embeddings 6h ...")
emb_tr6,  lbl_tr6  = extract_embeddings(extractor, loaders_6h["train"],  "train")
emb_va6,  lbl_va6  = extract_embeddings(extractor, loaders_6h["val"],    "val  ")
emb_te6,  lbl_te6  = extract_embeddings(extractor, loaders_6h["test"],   "test ")

log("Extrayendo embeddings 12h ...")
emb_tr12, lbl_tr12 = extract_embeddings(extractor, loaders_12h["train"], "train")
emb_va12, lbl_va12 = extract_embeddings(extractor, loaders_12h["val"],   "val  ")
emb_te12, lbl_te12 = extract_embeddings(extractor, loaders_12h["test"],  "test ")

print(f"Embeddings 6h  train: {emb_tr6.shape}  val: {emb_va6.shape}  test: {emb_te6.shape}")
print(f"Embeddings 12h train: {emb_tr12.shape}  val: {emb_va12.shape}  test: {emb_te12.shape}")

# --- Paso 2: entrenar MLP (segundos, no horas) ---
pw6  = float((lbl_tr6 == 0).sum()  / max((lbl_tr6 == 1).sum(),  1))
pw12 = float((lbl_tr12 == 0).sum() / max((lbl_tr12 == 1).sum(), 1))
print(f"pos_weight 6h={pw6:.2f}  12h={pw12:.2f}")

log("Entrenando MLP — horizonte 6h ...")
mlp_6h  = train_mlp(emb_tr6,  lbl_tr6,  emb_va6,  lbl_va6,  pw6)

log("Entrenando MLP — horizonte 12h ...")
mlp_12h = train_mlp(emb_tr12, lbl_tr12, emb_va12, lbl_va12, pw12)

[10:58:05] Extrayendo embeddings 6h ...



[10:59:19] Extrayendo embeddings 12h ...



Embeddings 6h  train: torch.Size([8000, 768])  val: torch.Size([6132, 768])  test: torch.Size([6050, 768])
Embeddings 12h train: torch.Size([8000, 768])  val: torch.Size([5420, 768])  test: torch.Size([5376, 768])
pos_weight 6h=13.47  12h=13.18
[11:00:27] Entrenando MLP — horizonte 6h ...


    Epoch  10 | val AUROC=0.7821


    Epoch  20 | val AUROC=0.7916


    Early stopping en epoch 25
    Mejor val AUROC: 0.7957
[11:00:29] Entrenando MLP — horizonte 12h ...


    Epoch  10 | val AUROC=0.8059


    Epoch  20 | val AUROC=0.8060


    Early stopping en epoch 29
    Mejor val AUROC: 0.8071


## PASO 6 — Evaluación

Se reportan cinco métricas, todas calculadas por `metricas.py` para que sean idénticas en los cinco cuadernos:

| Métrica | Qué mide | Mejor |
|---|---|---|
| AUROC | Capacidad de ordenar a los pacientes por riesgo | mayor |
| AUPRC | Lo mismo, centrado en la clase minoritaria (sepsis ≈ 8,6 %) | mayor |
| Sens@Esp90 | Sensibilidad en el punto de operación clínico (especificidad fija del 90 %) | mayor |
| Brier | Error cuadrático medio entre probabilidad predicha y etiqueta | menor |
| ECE | Desviación media entre probabilidad anunciada y frecuencia observada (10 bins por cuantiles) | menor |

**Recalibración de Platt.** El entrenamiento pondera la clase positiva
(`pos_weight` ≈ 11) para compensar el desbalanceo, lo que infla sistemáticamente las
probabilidades: sin corregir, el modelo anuncia riesgos varias veces superiores a la
prevalencia real y su Brier queda incluso por detrás del de un predictor que siempre
anunciara la prevalencia. `metricas_con_calibracion` ajusta el reescalado
$\sigma(a \cdot \mathrm{logit}(p) + b)$ **sobre la partición de validación** y lo
aplica después a ambas particiones. Por ser estrictamente monótono no altera el orden
de los pacientes: AUROC, AUPRC y Sens@Esp90 quedan intactos y solo cambian Brier y ECE.
Las claves `*_sin_calibrar` del resultado conservan los valores previos a la corrección,
y `platt_a` / `platt_b` los coeficientes ajustados.

El conjunto de prueba no interviene ni en el ajuste del calibrador ni en ninguna otra
decisión. Las probabilidades se persisten en `experiments/<modelo>/probs_*.npz` (crudas)
y `probs_cal_*.npz` (calibradas), de modo que añadir una métrica nueva más adelante no
obligue a reentrenar nada.


In [7]:
from metricas import metricas_con_calibracion, NOTA_METRICAS


def probs_mlp(mlp, emb):
    mlp.eval()
    with torch.no_grad():
        return torch.sigmoid(mlp(emb.to(DEVICE))).cpu().numpy()

print("=" * 78)
print("  HORIZONTE 6h — sistema con recalibración de Platt (ajustada en validación)")
print("=" * 78)
r6 = metricas_con_calibracion(lbl_va6.numpy(), probs_mlp(mlp_6h, emb_va6),
                              lbl_te6.numpy(), probs_mlp(mlp_6h, emb_te6),
                              etiqueta="Bio_ClinicalBERT 6h",
                              out_dir=OUT_DIR, horizonte="6h")
r6_val, r6_test = r6["val"], r6["test"]

print("\n" + "=" * 78)
print("  HORIZONTE 12h")
print("=" * 78)
r12 = metricas_con_calibracion(lbl_va12.numpy(), probs_mlp(mlp_12h, emb_va12),
                               lbl_te12.numpy(), probs_mlp(mlp_12h, emb_te12),
                               etiqueta="Bio_ClinicalBERT 12h",
                               out_dir=OUT_DIR, horizonte="12h")
r12_val, r12_test = r12["val"], r12["test"]


  HORIZONTE 6h — sistema con recalibración de Platt (ajustada en validación)
  Bio_ClinicalBERT 6h val [cal]       AUROC=0.7957  AUPRC=0.1736  Sens@Spec90=0.3030  Brier=0.0602  ECE=0.0086
  Bio_ClinicalBERT 6h test [cal]      AUROC=0.7998  AUPRC=0.1791  Sens@Spec90=0.3224  Brier=0.0603  ECE=0.0086
      Platt (ajustado en validación): a=0.6690  b=-2.4079  |  Brier prueba 0.2032 -> 0.0603  |  ECE 0.2731 -> 0.0086

  HORIZONTE 12h
  Bio_ClinicalBERT 12h val [cal]      AUROC=0.8071  AUPRC=0.1953  Sens@Spec90=0.3463  Brier=0.0608  ECE=0.0099
  Bio_ClinicalBERT 12h test [cal]     AUROC=0.8040  AUPRC=0.2034  Sens@Spec90=0.3495  Brier=0.0622  ECE=0.0127
      Platt (ajustado en validación): a=0.6209  b=-2.0661  |  Brier prueba 0.1601 -> 0.0622  |  ECE 0.2047 -> 0.0127


## PASO 7 — Exportar embeddings [CLS] alineados para la red de fusión

Se re-extraen y guardan los embeddings `[CLS]` (768-dim) de **todas** las notas de train/val/test para cada horizonte, en orden fijo (ordenado por `stay_id`) y con los `stay_id` **perfectamente alineados** con las filas de embeddings.

> **Corrección importante**: la versión anterior guardaba para *train* los 8.000 embeddings del submuestreo (barajados) junto a los ~28.500 `stay_id` del conjunto completo, de modo que al reconstruir el diccionario en la fusión (`zip(stay_ids, embs)`) los embeddings quedaban emparejados con `stay_id` equivocados. Aquí se re-extraen todas las notas en orden determinista, garantizando alineación y cobertura completa. El submuestreo a 8.000 sigue aplicándose **solo** para entrenar el MLP (PASO 5), no para la exportación a la fusión.

In [8]:
log("Exportando embeddings [CLS] ALINEADOS y de cobertura completa para la fusión ...")

# NOTA: emb_tr6/emb_tr12 se extrajeron sobre el submuestreo de 8.000 notas y en orden
# barajado (para entrenar el MLP), por lo que NO deben guardarse para la fusión.
# Aquí se re-extraen los embeddings de TODAS las notas de cada split, en orden fijo
# (ordenado por stay_id), con los stay_id perfectamente alineados con las filas de embeddings.

def export_split_embeddings(notes_df, split, tokenizer, extractor, max_len, batch_size=32):
    subset   = notes_df.filter(pl.col("split") == split).sort("stay_id")
    texts    = subset["note_text"].to_list()
    stay_ids = subset["stay_id"].to_numpy()
    labels   = subset["sepsis"].to_numpy().astype(np.int64)

    enc = tokenizer(texts, max_length=max_len, padding="max_length",
                    truncation=True, return_tensors="pt")
    extractor.eval()
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            ids  = enc["input_ids"][i:i + batch_size].to(DEVICE)
            mask = enc["attention_mask"][i:i + batch_size].to(DEVICE)
            embs.append(extractor(ids, mask).cpu().numpy())
            if (i // batch_size) % 20 == 0:
                print(f"    {split} {i}/{len(texts)} ...", end="\r")
    embs = (np.concatenate(embs, axis=0) if embs
            else np.zeros((0, 768), dtype=np.float32)).astype(np.float32)
    print()
    return stay_ids, embs, labels

for horizon, notes_df in [("6h", notes_6h), ("12h", notes_12h)]:
    for split in ["train", "val", "test"]:
        sids, embs, lbls = export_split_embeddings(
            notes_df, split, tokenizer, extractor, MAX_LEN
        )
        assert len(sids) == len(embs) == len(lbls), "desalineación stay_id/emb/label"
        np.save(OUT_DIR / f"cls_{horizon}_{split}.npy",      embs)
        np.save(OUT_DIR / f"labels_{horizon}_{split}.npy",   lbls)
        np.save(OUT_DIR / f"stay_ids_{horizon}_{split}.npy", sids)
        print(f"  {horizon} {split}: emb {embs.shape}  ids {sids.shape}  "
              f"sepsis={int(lbls.sum())}  (alineado, cobertura completa)")

[11:00:31] Exportando embeddings [CLS] ALINEADOS y de cobertura completa para la fusión ...


  6h train: emb (28406, 768)  ids (28406,)  sepsis=1966  (alineado, cobertura completa)



  6h val: emb (6132, 768)  ids (6132,)  sepsis=429  (alineado, cobertura completa)



  6h test: emb (6050, 768)  ids (6050,)  sepsis=425  (alineado, cobertura completa)


  12h train: emb (25175, 768)  ids (25175,)  sepsis=1775  (alineado, cobertura completa)



  12h val: emb (5420, 768)  ids (5420,)  sepsis=387  (alineado, cobertura completa)



  12h test: emb (5376, 768)  ids (5376,)  sepsis=392  (alineado, cobertura completa)


## PASO 8 — Guardar modelos y resultados

In [9]:
torch.save(mlp_6h.state_dict(),  OUT_DIR / "mlp_6h.pt")
torch.save(mlp_12h.state_dict(), OUT_DIR / "mlp_12h.pt")
# Extractor compartido (mismo para los dos horizontes)
torch.save(extractor.state_dict(), OUT_DIR / "bert_extractor.pt")

results = {
    "6h":  {"val": r6_val,  "test": r6_test},
    "12h": {"val": r12_val, "test": r12_test},
    "model_name": MODEL_NAME,
    "metricas_calibracion": NOTA_METRICAS,
    "approach": "frozen feature extractor + MLP",
    "max_len": MAX_LEN,
    "max_train_samples": MAX_TRAIN_SAMPLES,
    "mlp_arch": "768→256→64→1",
    "notes_6h":  {s: int(notes_6h.filter(pl.col("split") == s).shape[0])
                  for s in ["train", "val", "test"]},
    "notes_12h": {s: int(notes_12h.filter(pl.col("split") == s).shape[0])
                  for s in ["train", "val", "test"]},
}
with open(OUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
log(f"Completado — modelos en {OUT_DIR}/")

{
  "6h": {
    "val": {
      "auroc": 0.7957,
      "auprc": 0.1736,
      "sens_at_spec90": 0.303,
      "brier": 0.0602,
      "ece": 0.0086,
      "brier_sin_calibrar": 0.1996,
      "ece_sin_calibrar": 0.2684,
      "platt_a": 0.668951,
      "platt_b": -2.40794
    },
    "test": {
      "auroc": 0.7998,
      "auprc": 0.1791,
      "sens_at_spec90": 0.3224,
      "brier": 0.0603,
      "ece": 0.0086,
      "brier_sin_calibrar": 0.2032,
      "ece_sin_calibrar": 0.2731,
      "platt_a": 0.668951,
      "platt_b": -2.40794
    }
  },
  "12h": {
    "val": {
      "auroc": 0.8071,
      "auprc": 0.1953,
      "sens_at_spec90": 0.3463,
      "brier": 0.0608,
      "ece": 0.0099,
      "brier_sin_calibrar": 0.1514,
      "ece_sin_calibrar": 0.195,
      "platt_a": 0.620854,
      "platt_b": -2.066108
    },
    "test": {
      "auroc": 0.804,
      "auprc": 0.2034,
      "sens_at_spec90": 0.3495,
      "brier": 0.0622,
      "ece": 0.0127,
      "brier_sin_calibrar": 0.1601,
      "

## PASO 9 — Antigüedad del informe seleccionado

El informe asignado a cada stay es «el más reciente anterior a `ref_time`», **sin límite de
antigüedad**. Si buena parte de los informes fuesen antiguos, la contribución del texto medida
en la ablación *leave-one-out* sería una **cota inferior** de lo que aportaría la modalidad con
una ventana acotada. Este paso responde dos preguntas: cómo se reparte la antigüedad y si el
texto discrimina mejor cuando el informe es reciente.

El código está en `analisis_antiguedad_informe.py`. No vuelve a pasar BERT: reconstruye las
predicciones de test desde los embeddings `[CLS]` exportados en el PASO 7 y los pesos del MLP
del PASO 8, y comprueba que el AUROC global reproduce el de `results.json`.

> ⚠️ **Confusor que condiciona la lectura.** Los pacientes más graves reciben imagen con más
> frecuencia, de modo que *un informe reciente puede ser en sí mismo un marcador de gravedad*.
> Por eso la tabla reporta la **prevalencia** de cada tramo junto al AUROC: si ambas suben
> hacia los tramos recientes, el efecto está confundido y **no** demuestra que el texto
> «envejezca». Los tramos con menos de 30 positivos (o menos de 30 negativos) se marcan como
> no interpretables en lugar de reportar su AUROC.

In [10]:
exec(open("analisis_antiguedad_informe.py").read())

df_ant_6h  = analizar("6h",  notes_6h,  cohort_6h,  OUT_DIR, SepsisMLPClassifier, DEVICE)
df_ant_12h = analizar("12h", notes_12h, cohort_12h, OUT_DIR, SepsisMLPClassifier, DEVICE)
df_ant_6h.to_csv(OUT_DIR / "antiguedad_informe_6h.csv", index=False)
df_ant_12h.to_csv(OUT_DIR / "antiguedad_informe_12h.csv", index=False)

# ¿Suben AUROC y prevalencia a la vez? Si sí, el efecto está confundido.
from scipy.stats import spearmanr
for h, df in [("6h", df_ant_6h), ("12h", df_ant_12h)]:
    t = df[(df.seccion == "tramo") & (df.interpretable == True)]
    rho, _ = spearmanr(t["auroc"], t["prevalencia"])
    print(f"  {h}: Spearman(AUROC, prevalencia) = {rho:+.3f} "
          f"sobre {len(t)} tramos interpretables  |  "
          f"rango AUROC {t['auroc'].max() - t['auroc'].min():+.4f}, "
          f"rango prevalencia {t['prevalencia'].max() - t['prevalencia'].min():+.4f}")


=== Antigüedad del informe [6h] (n=40,588) ===
  p10       3.0 h
  p25       8.1 h
  p50      18.2 h
  p75      28.9 h
  p90      43.4 h
  p95      58.6 h
  media    22.2 h   máx 2519.8 h

  Distribución por tramo:
    ≤6h       7,777  ( 19.2 %)
    6–12h     6,333  ( 15.6 %)
    12–24h   12,452  ( 30.7 %)
    1–3d     12,884  ( 31.7 %)
    3–7d      1,056  (  2.6 %)
    >7d          86  (  0.2 %)

  AUROC global de test reconstruido 6h: 0.7998

=== AUROC por antigüedad del informe [BERT 6h] ===
  Tramo          n    pos     neg    prev    AUROC
  ------------------------------------------------
  ≤6h        1,141    106   1,035   9.3%   0.8210
  6–12h        968     72     896   7.4%   0.8091
  12–24h     1,876    109   1,767   5.8%   0.8026
  1–3d       1,891    108   1,783   5.7%   0.7723
  3–7d         152     20     132  13.2%      n/d
  >7d           22     10      12  45.5%      n/d

  n/d = menos de 30 positivos en el tramo; AUROC no interpretable.
  ⚠️ Compara la columna de p